<span style="font-family:monospace;letter-spacing:.14em;color:#857259">BANA · MODULE 3 · STATISTICAL REASONING FOR BUSINESS</span>

# Session 2: The logic of a test: could this gap be luck?

**Data:** EV purchase intention (Kaggle Playground Series S6E9, a *synthetic* dataset). `ev_m3_sample.csv` is the cleaned 10,000-row class sample from Module 2. `ev_m3_population.csv` has 50,000 rows that we treat as **everybody**, so we can see what sampling does.
**Time:** 60 minutes. **Course outcome:** CO3.

| Block | Min | What you do |
|---|---|---|
| Hook | 5 | The lady tasting tea: 8 out of 8 correct, luck or skill? |
| Concept | 10 | Null, alternative, p-value, errors (instructor) |
| Guided lab | 30 | A shuffle (permutation) test with no formulas |
| Team task | 10 | Write a null and an alternative for three business claims |
| Exit ticket | 5 | Three quick checks |

**By the end of this session you can:**
- state a null and an alternative hypothesis for a business claim
- explain a p-value as 'how often luck alone gives a gap this big'
- run a shuffle test in Python and read its result
- name the two kinds of mistake a test can make

**New commands this session (max 5):** `np.random.default_rng`, `rng.choice`, `rng.permutation`, `np.abs`, `(condition).mean()` as a p-value

**AI-use (AMBER protocol).** You may use an AI tool to look up *syntax*. The reasoning and interpretation must be your own: you will defend them in the viva.

In [ ]:
# -- Setup: run once ------------------------------------------------------
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
warnings.filterwarnings("ignore")

# House style: Nikhil D Jonathan design system (sand ground, terracotta / lagoon / ochre data colours)
C = {"lagoon": "#25707B", "terracotta": "#B5562F", "ochre": "#E0A231", "moss": "#3F6F4B",
     "earth": "#665643", "ink": "#221B15", "sand": "#F8F3EA", "card": "#FFFDF9", "line": "#E3D6C1"}
sns.set_theme(style="whitegrid", rc={
    "figure.facecolor": C["sand"], "axes.facecolor": C["card"], "axes.edgecolor": "#CDBCA3",
    "grid.color": C["line"], "text.color": C["ink"], "axes.labelcolor": C["ink"],
    "xtick.color": "#4A3E31", "ytick.color": "#4A3E31", "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 100, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlelocation": "left"})
PCT = PercentFormatter(1, decimals=0)                              # axis labels as percentages
SEQ = sns.light_palette(C["terracotta"], as_cmap=True)   # one-hue sequential map for heatmaps

from scipy import stats
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

# -- Load the data -------------------------------------------------------------
# Needs two files: ev_m3_sample.csv (10,000 rows: the cleaned Module 2 sample)
#                  ev_m3_population.csv (50,000 rows: we treat these as "everybody")
def find(name):
    for folder in [".", "data", "../data"]:
        if os.path.exists(os.path.join(folder, name)):
            return os.path.join(folder, name)
    try:
        from google.colab import files
        print("Choose", name); files.upload(); return name
    except ImportError:
        raise FileNotFoundError(f"Put {name} next to this notebook or in a data folder.")

def add_derived(d):
    """The four derived columns built in Session 1 (pre-written so every notebook stands alone)."""
    d = d.copy()
    d["buy"] = (d["Will_Buy_EV"] == "Yes").astype(int)
    d["age_band"] = pd.cut(d["Age"], bins=[24, 34, 44, 54, 69], labels=["25-34", "35-44", "45-54", "55-69"])
    d["income_band"] = pd.qcut(d["Annual_Income_USD"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
    d["charger_total"] = d["Charging_Stations_Near_Home"] + d["Charging_Stations_Near_Work"]
    return d

df = add_derived(pd.read_csv(find("ev_m3_sample.csv")))        # the class sample
pop = add_derived(pd.read_csv(find("ev_m3_population.csv")))   # "everybody"
print("Sample:", len(df), "rows | Population:", len(pop), "rows")

## Part A. Hook: the lady tasting tea (5 min)
A woman says she can tell whether milk or tea was poured first. Eight cups: four of each, in random order. She identifies **all eight correctly**.
**Predict first:** if she were only guessing, how likely is that? Write a number (1 in what?).

In [ ]:
rng = np.random.default_rng(2026)
truth = {0, 1, 2, 3}                                             # cups 0-3 really had milk first
matches = [len(truth & set(rng.____(8, 4, replace=False))) for _ in range(10000)]   # TODO: pick 4 cups at random (choice)
share_all_right = np.mean([m == 4 for m in matches])             # all four milk-first cups picked = all 8 correct
print(f"Out of 10,000 pure guessers, {share_all_right:.2%} got all eight right (about 1 in {1 / share_all_right:.0f})")

**YOUR ANSWER:** That share is the *p-value* of this experiment. If you were the host, would you believe her? What if she had got 7 of 8?

> _write here_

## Part B. Our question: do buyers earn more? (5 min)

In [ ]:
income = df["Annual_Income_USD"].values
label = df["buy"].values                                         # 1 = Yes, 0 = No
observed = income[label == 1].mean() - income[label == 0].mean()
print(f"Buyers earn {observed:,.0f} more than non-buyers, on average")

**Write the hypotheses.**
- **Null (H0):** 
- **Alternative (H1):** 

> _write here_

## Part C. Shuffle the labels (10 min)
If buying had *nothing* to do with income, the Yes/No tags would be random stickers. We shuffle the stickers 2,000 times and record the income gap each time.
**Predict first:** how big will the biggest shuffled gap be compared with the real gap of about 17,800?

In [ ]:
rng = np.random.default_rng(2026)
shuffled_gaps = []
for _ in range(2000):
    shuffled = rng.____(label)                                   # TODO: shuffle the labels (permutation)
    shuffled_gaps.append(income[shuffled == 1].mean() - income[shuffled == 0].mean())

fig, ax = plt.subplots(figsize=(8, 3.2))
sns.histplot(shuffled_gaps, binwidth=100, color=C["lagoon"], ax=ax)
ax.axvline(observed, color=C["terracotta"], lw=2)
ax.set_xlabel("Income gap (Yes minus No) when labels are shuffled; red line = the real gap"); ax.set_ylabel("")
ax.set_xlim(-1000, observed * 1.1)
plt.tight_layout()

**YOUR ANSWER:** Where does the real gap fall compared with the shuffled gaps? What does that say about luck?

> _write here_

## Part D. The p-value (5 min)
The **p-value** is the share of shuffles that produced a gap *at least as big* (in either direction) as the real one.

In [ ]:
p_value = (np.abs(shuffled_gaps) >= ____).mean()                # TODO: compare with the real gap (absolute value)
print(f"p-value = {p_value}  ({int(p_value * 2000)} of 2,000 shuffles reached the real gap)")
print("Write it as: p < 1/2000 = 0.0005" if p_value == 0 else "")

## Part E. Reuse the test on other questions (10 min)
The shuffle test is wrapped in a function. You only choose the column and the groups.

In [ ]:
def shuffle_test(values, label, reps=2000, seed=2026):
    """Gap in the mean of `values` between label==1 and label==0, and the shuffle p-value."""
    rng = np.random.default_rng(seed)
    values, label = np.asarray(values), np.asarray(label)
    real = values[label == 1].mean() - values[label == 0].mean()
    gaps = []
    for _ in range(reps):
        s = rng.permutation(label)
        gaps.append(values[s == 1].mean() - values[s == 0].mean())
    return real, (np.abs(gaps) >= abs(real)).mean()

for col in ["Age", "Annual_Income_USD", "Daily_Commute_km"]:
    gap, p_ = shuffle_test(df[col], df["buy"])
    print(f"{col:20s} gap (Yes minus No) = {gap:>10,.1f}   p = {p_:.4f}")

In [ ]:
# Rural vs Urban: is the difference in Yes rate real?
ru = df[df["City_Type"].isin(["Rural", "Urban"])]
gap, p_ = shuffle_test(ru["____"], (ru["City_Type"] == "Rural").astype(int))     # TODO: which column holds Yes = 1, No = 0?
print(f"Rural minus Urban Yes rate = {gap:.1%}   p = {p_:.4f}")

**YOUR ANSWER:** Which gaps are bigger than luck, and which are not? A p-value tells you if a gap is *real*. Does it tell you if the gap is *big*? (Hold that for Session 4.)

> _write here_

## Part F. Two ways to be wrong (reading, no code)
| | Truth: nothing going on | Truth: something is going on |
|---|---|---|
| **Test says "real"** | **Type I error**: false alarm (we act on luck) | Correct |
| **Test says "no evidence"** | Correct | **Type II error**: missed it (we ignore a real effect) |

The significance level **alpha = 0.05** means we accept a false alarm about 1 time in 20. A p-value below alpha is called *statistically significant*.

## Team task (10 min): null, alternative, guess, test
For each claim, write H0 and H1, **predict** whether the p-value will be small or large, then test it with `shuffle_test` (columns: `buy`, `Age`, `Daily_Commute_km`, `Gender`).
1. *"Older customers are more likely to buy."* (Age)
2. *"Long commuters are less likely to buy."* (Daily_Commute_km)
3. *"Gender affects buying."* (Gender: use Male and Female only)

In [ ]:
mf = df[df["Gender"].isin(["Male", "Female"])]
gap, p_ = shuffle_test(mf["buy"], (mf["Gender"] == "____").astype(int))          # TODO: group 1 = Male
print(f"Male minus Female Yes rate = {gap:.2%}   p = {p_:.3f}")

## Exit ticket (5 min)
1. A p-value is 0.03. Finish the sentence: "If nothing were going on, then ...".
2. Is a p-value the probability that the null hypothesis is true? Say yes or no and why.
3. Name one cost of a Type I error and one cost of a Type II error for an EV subsidy decision.